# SageMaker Feature Store — Getting Started

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.0.0</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.7.1</strong></li>
</ul>
</div>

### Install packages
Please choose `Python 3 (ipykernel)` kernel to proceed.

We will first install the prerequisite packages. They should be already installed if you run the [Setup.ipynb](../Setup.ipynb) notebook. If you experience problems, uncomment the following three cells to re-install the right libraires and the restart the kernel and then continue

In [ ]:
## Uncomment and execute if having errors import any of the sagemaker libraries
# %pip install --no-cache-dir sagemaker-core==2.7.1 \
#     sagemaker-train==1.7.1 \
#     sagemaker-serve==1.7.1 --extra-index-url https://download.pytorch.org/whl/cpu \
#     sagemaker-mlops==1.7.1 \
#     sagemaker==3.7.1

In [ ]:
# # Restart kernel to pick up the updated packages
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

The core value is solving the "feature reuse" problem. Without a feature store, every team and every project re-engineers the same features from scratch, and training vs inference often use different feature computation code — which causes training-serving skew.
```
Feature Store (service)
  └── Feature Group (like a table)
        ├── Feature Definitions (columns + types)
        ├── Record Identifier (primary key column)
        ├── Event Time (timestamp column)
        ├── Online Store (latest records, key-value)
        └── Offline Store (all records, S3 Parquet)
              ├── S3 bucket (your data)
              └── Glue Data Catalog table (for Athena queries)

Record = one row of feature values for one identifier at one event time
Feature = one column (name + type: Integral, Fractional, or String)
```


Why Feature Store:

**Single source of truth** — Features are computed once, stored centrally, and reused across models and teams. No more copy-pasting feature engineering code between notebooks.

**Training-serving consistency** — The online store serves the exact same features at inference time that the offline store provided during training. Same data, same schema, no skew.

**Two access patterns from one ingestion** — You `put_record` once and get both a low-latency online store (for real-time predictions) and an S3-backed offline store (for batch training). No need to maintain two separate data pipelines.

**Time-travel and auditability** — The offline store keeps every version of every record. 

**Discovery and sharing** — Feature groups are registered in the Glue catalog with metadata. 



## The online store and offline store are backed by different patterns:

**Online Store** — Uses a managed key-value store (think DynamoDB-style). There are two tiers:
- *Standard* — managed low-latency store, single-digit millisecond reads
- *InMemory* — backed by ElastiCache (Redis OSS) for sub-millisecond reads at high throughput
It keeps the latest record per **identifier**. (Record identifier/ Event time)

**Offline Store** — S3 + Glue + Athena:
- Data lands as Parquet files in your S3 bucket
- A Glue Data Catalog table is auto-created so Athena can query it


**Ingestion path** — When you call `PutRecord`, it writes to the online store synchronously. The offline store gets updated asynchronously — records are batched and written to S3.

### Do you need both stores? No. 
Online only — real-time serving, no historical data
Offline only — training/batch use, no real-time lookup
Both — gives you the full picture

A simple, step-by-step notebook that creates a Feature Group,
ingests sample data, and reads it back from the online store.

Each step maps to a numbered section so you can follow along easily.

## 1. Setup

In [ ]:
import boto3
import pandas as pd
import time
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name

# get_execution_role() works in SageMaker Studio.
# Locally, fall back to a known role.
try:
    role = get_execution_role()
except ValueError:
    role = f"arn:aws:iam::{boto3.client('sts').get_caller_identity()['Account']}:role/SageMakerExecutionRole"
    print(f"(Running locally — using role: {role})")

sm_client = boto3.client("sagemaker", region_name=region)
fs_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

feature_group_name = "xx-feature-group-" + time.strftime("%d-%H-%M-%S")
print(f"Region:        {region}")
print(f"Feature Group: {feature_group_name}")

## 2. Prepare Sample Data

Every feature group needs:
- A **record identifier** — unique ID per row
- An **event time** — ISO-8601 timestamp string

In [ ]:
df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "age":         [25, 34, 45, 28, 52],
    "income":      [50000.0, 72000.0, 95000.0, 61000.0, 110000.0],
    "is_premium":  [0, 1, 1, 0, 1],
    "event_time":  [time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())] * 5,
})
df

## 3. Build Feature Definitions

Feature Store supports 3 types: `Integral`, `Fractional`, `String`.
We map pandas dtypes automatically.

In [ ]:
def get_feature_type(dtype):
    if dtype in ("int64", "int32"):
        return "Integral"
    elif dtype in ("float64", "float32"):
        return "Fractional"
    return "String"

feature_definitions = [
    {"FeatureName": col, "FeatureType": get_feature_type(str(df[col].dtype))}
    for col in df.columns
]

for fd in feature_definitions:
    print(f"  {fd['FeatureName']:20s} -> {fd['FeatureType']}")

## 4. Create the Feature Group

This creates a feature group with:
- **Online store** — low-latency reads for real-time inference
- **Offline store** — historical data in S3 (Parquet) for training

In [ ]:
sm_client.create_feature_group(
    FeatureGroupName=feature_group_name,
    RecordIdentifierFeatureName="customer_id",
    EventTimeFeatureName="event_time",
    FeatureDefinitions=feature_definitions,
    OnlineStoreConfig={"EnableOnlineStore": True},
    OfflineStoreConfig={
        "S3StorageConfig": {
            "S3Uri": f"s3://{bucket}/feature-store-demo/offline"
        }
    },
    RoleArn=role,
)

# Wait until it's ready
print("Waiting for feature group to be created...")
while True:
    status = sm_client.describe_feature_group(
        FeatureGroupName=feature_group_name
    )["FeatureGroupStatus"]
    if status != "Creating":
        break
    time.sleep(5)
print(f"Feature Group status: {status}")

## 5. Ingest Records

We use `put_record` to write each row into the feature group.
This populates both the online and offline stores.

In [ ]:
for _, row in df.iterrows():
    record = [
        {"FeatureName": col, "ValueAsString": str(row[col])}
        for col in df.columns
    ]
    fs_runtime.put_record(
        FeatureGroupName=feature_group_name,
        Record=record,
    )
print(f"Ingested {len(df)} records into {feature_group_name}")

## 6. Read Back from the Online Store

In [ ]:
response = fs_runtime.get_record(
    FeatureGroupName=feature_group_name,
    RecordIdentifierValueAsString="1",
)

print("Record for customer_id=1:")
for feat in response["Record"]:
    print(f"  {feat['FeatureName']:20s} = {feat['ValueAsString']}")

## 7. Offline Store Info

The offline store writes Parquet files to S3 and registers a table
in the AWS Glue Data Catalog. You can query it with Athena.

In [ ]:
desc = sm_client.describe_feature_group(FeatureGroupName=feature_group_name)
offline_s3 = desc["OfflineStoreConfig"]["S3StorageConfig"].get("ResolvedOutputS3Uri", "N/A")
catalog = desc["OfflineStoreConfig"].get("DataCatalogConfig", {})

print(f"Glue database:     {catalog.get('Database', 'N/A')}")
print(f"Glue table:        {catalog.get('TableName', 'N/A')}")

In [ ]:
import statistics

# ── Single GetRecord latency ──
get_times = []
for _ in range(20):
    start = time.time()
    fs_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString="1",
    )
    get_times.append((time.time() - start) * 1000)  # ms

print("── GetRecord (single) ──")
print(f"  Min:    {min(get_times):.1f} ms")
print(f"  Median: {statistics.median(get_times):.1f} ms")
print(f"  Mean:   {statistics.mean(get_times):.1f} ms")
print(f"  Max:    {max(get_times):.1f} ms")
print(f"  P95:    {sorted(get_times)[int(len(get_times)*0.95)]:.1f} ms")

In [ ]:
# ── BatchGetRecord latency ──
batch_times = []
for _ in range(20):
    start = time.time()
    fs_runtime.batch_get_record(
        Identifiers=[{
            "FeatureGroupName": feature_group_name,
            "RecordIdentifiersValueAsString": ["1", "2", "3", "4", "5"],
        }]
    )
    batch_times.append((time.time() - start) * 1000)

print("── BatchGetRecord (5 records) ──")
print(f"  Min:    {min(batch_times):.1f} ms")
print(f"  Median: {statistics.median(batch_times):.1f} ms")
print(f"  Mean:   {statistics.mean(batch_times):.1f} ms")
print(f"  Max:    {max(batch_times):.1f} ms")

## 8. Cleanup

In [ ]:
sm_client.delete_feature_group(FeatureGroupName=feature_group_name)
print(f"Deleted feature group: {feature_group_name}")